# Autoregressive MLP 训练

复用统一训练流水线，训练固定 Markov 上下文和权重共享输出的 MLP。

## 配置

In [ ]:
from datetime import datetime
from pathlib import Path
from scripts.experiment import (
    AutoregressiveMLPConfig,
    DataConfig,
    DeviceConfig,
    ExperimentConfig,
    SchedulerConfig,
    TrainingConfig,
)
from scripts.pipeline import (
    load_model_tier,
    model_summary,
    plot_loss,
    run_training_experiment,
)

model_tier = load_model_tier("mlp", "medium")
config = ExperimentConfig(
    data=DataConfig(
        dataset_path=Path("data/processed"),
        tokenizer_path=Path("data/processed/tokenizer.json"),
        output_dir=Path(model_tier["output_path"]),
    ),
    model=AutoregressiveMLPConfig(
        tau=model_tier["tau"],
        embedding_dim=model_tier["embedding_dim"],
        hidden_size=model_tier["hidden_size"],
    ),
    training=TrainingConfig(
        batch_size=512,
        num_epochs=50,
        learning_rate=1e-3,
        scheduler=SchedulerConfig(name="cosine", eta_min=1e-6),
    ),
    device=DeviceConfig(mode="formal_cuda", cuda_index=0),
)
model_summary(config)

## 训练和保存

In [ ]:
run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_log_dir = (
    Path("/root/tf-logs/passmini/training")
    / config.data.output_dir.relative_to("output")
    / run_id
)
artifacts = run_training_experiment(
    config,
    tensorboard_log_dir=tensorboard_log_dir,
    tensorboard_log_interval=100,
)
print(f"TensorBoard logs: {tensorboard_log_dir}")

In [ ]:
artifacts.history

## 测试损失和生成示例

In [ ]:
print(f"Test loss: {artifacts.test_loss:.4f}")
plot_loss(
    artifacts.history,
    test_loss=artifacts.test_loss,
    save_path=config.data.output_dir / "loss.png",
)

In [ ]:
artifacts.model.generate(max_length=12)